In [1]:
import numpy as np


def split_into_four_row_blocks(A: np.ndarray) -> list[np.ndarray]:
    """
    Split matrix A into 4 row-blocks, as evenly as possible.

    Parameters
    ----------
    A : ndarray, shape (m, n)
        Input matrix.

    Returns
    -------
    blocks : list of 4 ndarrays
        Each block is a subset of rows of A. The blocks are in order
        so that stacking them vertically recovers A.
    """
    # Using numpy to mimic Matlab "row sub-indexing" style:
    # A(i:j, :) <-> slices in axis=0
    blocks = np.array_split(A, 4, axis=0)
    if len(blocks) != 4:
        # This shouldn't happen with np.array_split(…, 4),
        # but we keep the check for safety.
        raise RuntimeError("Expected 4 row blocks from np.array_split.")
    return blocks


def tsqr_four_blocks(A: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Perform a TSQR factorisation of a tall–skinny matrix A,
    using FOUR row blocks (serial / no parallel).

    Algorithm (matching the lectures):
    1. Split A into 4 row blocks A1, A2, A3, A4.
    2. For each Ai, compute local QR:  Ai = Qi * Ri.
    3. Stack the four Ri vertically:
           R_stack = [R1; R2; R3; R4].
       Compute QR of this small stacked matrix:
           R_stack = Q_top * R_final.
    4. Partition Q_top into 4 blocks Q_top_i (by rows).
    5. Form the global Q as block-diagonal product:
           Qi_global = Qi * Q_top_i,
       and stack all Qi_global vertically.

    Parameters
    ----------
    A : ndarray, shape (m, n)
        Input tall–skinny matrix.

    Returns
    -------
    Q : ndarray, shape (m, n)
        Thin Q factor (columns are approximately orthonormal).
    R : ndarray, shape (n, n)
        Upper triangular R factor.
    """
    m, n = A.shape
    if m < 4:
        raise ValueError("Need at least 4 rows to split into 4 blocks.")

    # -----------------------------
    # 1. Split into 4 row blocks
    # -----------------------------
    blocks = split_into_four_row_blocks(A)  # list of 4 blocks

    # -----------------------------
    # 2. Local QR on each block
    #    Ai = Qi * Ri
    # -----------------------------
    Q_local: list[np.ndarray] = []
    R_local: list[np.ndarray] = []

    for B in blocks:
        # "reduced" QR: Q has shape (rows_of_B, n), R has shape (n, n)
        Qb, Rb = np.linalg.qr(B, mode="reduced")
        Q_local.append(Qb)
        R_local.append(Rb)

    # -----------------------------
    # 3. Top-level QR on stacked R
    #    [R1; R2; R3; R4] = Q_top * R_final
    # -----------------------------
    R_stacked = np.vstack(R_local)  # shape (4n, n)
    Q_top, R_final = np.linalg.qr(R_stacked, mode="reduced")  # Q_top: (4n, n)

    # -----------------------------
    # 4. Partition Q_top into 4 row blocks
    #    each Q_top_i has shape (n, n)
    # -----------------------------
    Q_top_blocks = np.split(Q_top, 4, axis=0)

    # -----------------------------
    # 5. Form global Q
    #    Qi_global = Qi_local * Q_top_i
    # -----------------------------
    Q_blocks_global: list[np.ndarray] = []
    for Qi_local, Qi_top in zip(Q_local, Q_top_blocks):
        # (m_i x n) @ (n x n) -> (m_i x n)
        Q_blocks_global.append(Qi_local @ Qi_top)

    Q_global = np.vstack(Q_blocks_global)  # (m, n)

    return Q_global, R_final


# ============================================================
# Helper: simple test & diagnostics
# ============================================================
def test_tsqr_once(m: int, n: int, seed: int | None = None) -> None:
    """
    Generate a random m x n tall–skinny matrix (m >> n),
    run TSQR, and print diagnostics comparing to NumPy's qr.
    """
    if seed is not None:
        rng = np.random.default_rng(seed)
    else:
        rng = np.random.default_rng()

    A = rng.standard_normal(size=(m, n))

    # Our TSQR
    Q_tsqr, R_tsqr = tsqr_four_blocks(A)

    # Reference QR from NumPy
    Q_ref, R_ref = np.linalg.qr(A, mode="reduced")

    # Check factorisation error: ||A - Q R|| / ||A||
    rel_err_tsqr = np.linalg.norm(A - Q_tsqr @ R_tsqr) / np.linalg.norm(A)
    rel_err_ref = np.linalg.norm(A - Q_ref @ R_ref) / np.linalg.norm(A)

    # Check orthogonality of Q: ||Q^T Q - I||
    I = np.eye(n)
    orth_err_tsqr = np.linalg.norm(Q_tsqr.T @ Q_tsqr - I)
    orth_err_ref = np.linalg.norm(Q_ref.T @ Q_ref - I)

    print("=== TSQR test (m = {}, n = {}) ===".format(m, n))
    print(f"NumPy  QR: rel factorisation error = {rel_err_ref: .3e}, "
          f"orthogonality error = {orth_err_ref: .3e}")
    print(f"TSQR    : rel factorisation error = {rel_err_tsqr: .3e}, "
          f"orthogonality error = {orth_err_tsqr: .3e}")
    print()


def run_tsqr_tests():
    """
    Run a few TSQR tests for different sizes.
    This is just for sanity checking.
    """
    sizes = [
        (40, 6),
        (80, 8),
        (120, 10),
    ]
    for (m, n) in sizes:
        test_tsqr_once(m=m, n=n, seed=42)

if __name__ == "__main__":
    # Example usage: run a few consistency tests.
    run_tsqr_tests()

=== TSQR test (m = 40, n = 6) ===
NumPy  QR: rel factorisation error =  3.586e-16, orthogonality error =  1.184e-15
TSQR    : rel factorisation error =  2.988e-16, orthogonality error =  5.885e-16

=== TSQR test (m = 80, n = 8) ===
NumPy  QR: rel factorisation error =  2.421e-16, orthogonality error =  9.841e-16
TSQR    : rel factorisation error =  3.556e-16, orthogonality error =  1.139e-15

=== TSQR test (m = 120, n = 10) ===
NumPy  QR: rel factorisation error =  2.090e-16, orthogonality error =  1.420e-15
TSQR    : rel factorisation error =  4.060e-16, orthogonality error =  1.430e-15



In [2]:
m, n = 60, 8
A = np.random.randn(m, n)
Q, R = tsqr_four_blocks(A)
print("Relative error:", np.linalg.norm(A - Q @ R) / np.linalg.norm(A))

Relative error: 3.855710779182826e-16


We implemented a serial version of the communication-avoiding TQSR algorithm by:
1. Splitting the matrix into 4 row blocks;
2. Performing local QR factorisations;
3. Performing a second QR on the stacked Rfactors;
4. Forming the final Q via block multiplication;
The relative factorisation errors are on the order of 10^(-6), which confirms numerical correctness.